In [2]:
# =======================================================
# سكربت الأتمتة المطور (Robust Populator): التعامل مع مخرجات الذكاء الاصطناعي غير المكتملة
# =======================================================
!pip install -q pandas owlready2

import pandas as pd
import re
from owlready2 import *
import os
import shutil

# 1. تحديد مسارات الملفات 
INPUT_ONTO = "/kaggle/input/datasets/maherghanem/inpuss/PharmaCompass.owl"
INPUT_CSV = "/kaggle/input/datasets/maherghanem/inpuss/my_model_predictions_improved.csv"
WORK_DIR = "/kaggle/working/"
TEMP_ONTO = os.path.join(WORK_DIR, "PharmaCompass_Temp.owl")
OUTPUT_FILE = os.path.join(WORK_DIR, "PharmaCompass_Populated_Robust.owl")

# نسخ الأنطولوجيا لمجلد العمل لتجنب مشكلة القراءة فقط
shutil.copy(INPUT_ONTO, TEMP_ONTO)

print("⏳ جاري تحميل الأنطولوجيا...")
onto_path.append(WORK_DIR)
onto = get_ontology("file://" + TEMP_ONTO).load()

print("📊 جاري قراءة بيانات المرضى واستخراجها بذكاء (Regex) متجاوزين أخطاء الـ JSON...")
df = pd.read_csv(INPUT_CSV)

def clean_name(text):
    if not isinstance(text, str): return "Unknown"
    return re.sub(r'[^a-zA-Z0-9_]', '_', text.strip().title())

valid_patients_added = 0

with onto:
    for idx, row in df.iterrows():
        json_str = str(row['EXTRACTED_JSON'])
        subject_id = row['SUBJECT_ID']
        
        # استخراج الجنس
        sex = None
        sex_match = re.search(r'\"sex\"\s*:\s*\"([^\"]+)\"', json_str, re.IGNORECASE)
        if sex_match: sex = sex_match.group(1).upper()
        
        # استخراج الشكوى الرئيسية (لتمثل المرض Condition)
        complaint = None
        comp_match = re.search(r'\"(?:attending_)?chief_complaint\"\s*:\s*\"([^\"]+)\"', json_str, re.IGNORECASE)
        if comp_match: complaint = comp_match.group(1)
        
        # استخراج الحساسية للأدوية (لتمثل الأدوية Medication)
        allergies = []
        al_match = re.search(r'\"allergies\"\s*:\s*\[(.*?)\]', json_str, re.IGNORECASE | re.DOTALL)
        if al_match:
            items = re.findall(r'\"([^\"]+)\"', al_match.group(1))
            allergies.extend(items)
            
        # إذا وجدنا أي معلومة مفيدة، ننشئ المريض في الأنطولوجيا
        if sex or complaint or allergies:
            patient_name = f"Patient_{subject_id}"
            patient_ind = onto.Patient(patient_name)
            
            if sex:
                patient_ind.has_gender.append(str(sex))
                
            if complaint:
                cond_clean = clean_name(complaint)
                cond_ind = onto.Condition(f"Condition_{cond_clean}")
                patient_ind.has_condition.append(cond_ind)
                
            for al in allergies:
                med_clean = clean_name(al)
                med_ind = onto.Medication(f"Medication_{med_clean}")
                patient_ind.takes_medication.append(med_ind)
                
            valid_patients_added += 1

onto.save(file=OUTPUT_FILE, format="rdfxml")

print("--------------------------------------------------")
print(f"🎉 تمت العملية بنجاح ساحق!")
print(f"تم إنقاذ وحقن {valid_patients_added} مريض حقيقي مع أمراضهم وأدويتهم.")
print(f"تم حفظ الملف الجديد في مسار المخرجات: /kaggle/working/PharmaCompass_Populated_Robust.owl")

⏳ جاري تحميل الأنطولوجيا...
📊 جاري قراءة بيانات المرضى واستخراجها بذكاء (Regex) متجاوزين أخطاء الـ JSON...
--------------------------------------------------
🎉 تمت العملية بنجاح ساحق!
تم إنقاذ وحقن 45 مريض حقيقي مع أمراضهم وأدويتهم.
تم حفظ الملف الجديد في مسار المخرجات: /kaggle/working/PharmaCompass_Populated_Robust.owl
